# Cost Analysis

We profile and benchmark the jit-ed cost function.

In [ ]:
%matplotlib ipympl

import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
import jax.numpy as jnp
import numpy as np

import exp_mpc.stewart_min.const as const
import exp_mpc.stewart_min.opt as opt
import exp_mpc.stewart_min.quartic_cost as quartic_cost


In [ ]:
n = 200  # horizon

acc_ref = jnp.array([0.3, -0.2, 0.1]) + const.gravity
acc_ref = jnp.tile(A=acc_ref, reps=(n, 1))
omega_ref = jnp.array([-0.1, 0.2, -0.3])
omega_ref = jnp.tile(A=omega_ref, reps=(n, 1))

In [ ]:
weights = opt.ExpWeights(
    acc=jnp.array([5., 5., 1.]),
    omega=jnp.array([1., 1., 5.]),
    alpha_acc=jnp.array([1.]),
    alpha_omega=jnp.array([4.]),
)


margins = [0.2, 0.1]
sizes = [1.0, 2**3, 2**8]

leg_cost = quartic_cost.QuarticCost.from_bounds(
    margins=[0.3, 0.2, 0.1],
    sizes=[1.0, 2**3, 2**5, 2**10],
    # margins=margins,
    # sizes=sizes,
    low=const.leg_min,
    high=const.leg_max,
)
leg_vel_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    # sizes=[size * 2**6 for size in sizes],
    sizes = sizes,
    low=-const.max_leg_vel,
    high=const.max_leg_vel,
)
joint_angle_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.joint_max_angle,
    high=const.joint_max_angle,
)
yaw_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.max_yaw,
    high=const.max_yaw,
)
cost_terms = opt.CostTerms(
    leg_cost=leg_cost,
    leg_vel_cost=leg_vel_cost,
    joint_angle_cost=joint_angle_cost,
    yaw_cost=yaw_cost,
)

In [ ]:
np.random.seed(42)

rstate0 = jnp.zeros(12)
vstate0_irl = jnp.zeros(15)
vstate0_sim = jnp.zeros(15)
control0 = jnp.zeros(6)
control_flat = np.random.uniform(-1, 1, n * 6)

In [ ]:
def call0() -> tuple[jax.Array, jax.Array]:
    res = opt.cost_and_grad_flat_jax(
        weights=weights,
        cost_terms=cost_terms,
        acc_ref=acc_ref,
        omega_ref=omega_ref,
        rstate0=rstate0,
        vstate0_irl=vstate0_irl,
        vstate0_sim=vstate0_sim,
        control0=control0,
        control_flat=control_flat,
    )
    res[0].block_until_ready()
    res[1].block_until_ready()
    return res

def call1() -> jax.Array:
    res = opt.cost_flat_jax(
        weights=weights,
        cost_terms=cost_terms,
        acc_ref=acc_ref,
        omega_ref=omega_ref,
        rstate0=rstate0,
        vstate0_irl=vstate0_irl,
        vstate0_sim=vstate0_sim,
        control0=control0,
        control_flat=control_flat,
    )
    res.block_until_ready()
    return res

In [ ]:
call0()
%timeit -n 1000 -r 7 call0()

In [ ]:
call1()
%timeit -n 1000 -r 7 call1()

In [ ]:
# call0()
# with jax.profiler.trace("/tmp/jax-trace", create_perfetto_link=True):
#     call0()

In [ ]:
# call1()
# with jax.profiler.trace("/tmp/jax-trace", create_perfetto_link=True):
#     call1()